# 1
# Notebook 01: Data Collection

## Purpose
Pull resolved markets and trade histories from Polymarket and Kalshi.
Save to parquet files for use in NB 02 (feature engineering).

## Outputs
- `data/raw/markets_polymarket.parquet`
- `data/raw/markets_kalshi.parquet`
- `data/raw/trades_polymarket.parquet`
- `data/raw/trades_kalshi.parquet`

In [20]:
# 2
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time
import json

# Create data directories if they don't exist
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

print("Directories ready.")

Directories ready.


In [51]:
# 3
def fetch_polymarket_markets(limit=500, max_pages=50):
    """
    Fetch resolved markets from Polymarket Gamma API.
    No auth required. Returns a list of market dicts.
    """
    base_url = "https://gamma-api.polymarket.com/markets"
    all_markets = []
    offset = 0

    for _ in tqdm(range(max_pages), desc="Fetching Polymarket markets"):
        params = {
            "closed": "true",
            "limit": limit,
            "offset": offset
        }
        resp = requests.get(base_url, params=params)
        resp.raise_for_status()
        data = resp.json()

        if not data:
            break

        all_markets.extend(data)
        offset += limit
        time.sleep(0.5)  # be polite to the API

        if len(data) < limit:
            break

    print(f"Fetched {len(all_markets)} Polymarket markets")
    return all_markets

pm_markets_raw = fetch_polymarket_markets()

Fetching Polymarket markets:   0%|          | 0/50 [00:01<?, ?it/s]

Fetched 100 Polymarket markets


In [52]:
# Cell 4 — Parse and save Polymarket markets
def parse_polymarket_market(m):
    return {
        "market_id": m.get("conditionId"),
        "question": m.get("question"),
        "category": m.get("category"),
        "start_date": m.get("startDate"),
        "end_date": m.get("endDate"),
        "resolution_source": m.get("resolutionSource"),
        "volume": m.get("volume"),
        "liquidity": m.get("liquidity"),
        "outcome": m.get("outcome"),
        "winning_outcome": m.get("outcomePrices"),
        "platform": "polymarket"
    }

pm_markets = pd.DataFrame([parse_polymarket_market(m) for m in pm_markets_raw])
pm_markets["volume"] = pd.to_numeric(pm_markets["volume"], errors="coerce")

pm_markets.to_parquet("../data/raw/markets_polymarket.parquet", index=False)
print(f"Total Polymarket markets: {len(pm_markets)}")
pm_markets.head()

Total Polymarket markets: 100


,market_id,question,category,start_date,end_date,resolution_source,volume,liquidity,outcome,winning_outcome,platform
0,0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa0...,Will Joe Biden get Coronavirus before the elec...,US-current-affairs,None,2020-11-04T00:00:00Z,None,32257.445115,0,None,"[""0"", ""0""]",polymarket
1,0x44f10d1cd5aaed4b7ae0b5edb76790f54f45dc0bcaa8...,Will Airbnb begin publicly trading before Jan ...,Tech,None,2021-01-02T00:00:00Z,None,89665.252158,0,None,"[""0"", ""0""]",polymarket
2,0x3e0524de013d9dc359f5eb370773f25de2f03d320029...,Will a new Supreme Court Justice be confirmed ...,US-current-affairs,None,2020-11-04T00:00:00Z,None,43279.456005,0,None,"[""0"", ""0""]",polymarket
3,0x9b946f54f3428aafc308c33aa04a943fe13a011bdac9...,Will Kim Kardashian and Kanye West divorce bef...,Pop-Culture,2023-08-24T16:35:24.938Z,2021-01-02T00:00:00Z,None,22067.475119,0.179651,None,"[""0.000001011082052522541417308141468657552"", ...",polymarket
4,0x5d1a1ab716fd06943441fe27cde0089651ce769bec55...,Will Coinbase begin publicly trading before Ja...,Crypto,None,2021-01-02T00:00:00Z,None,116803.377183,0.367501,None,"[""0.000001024519509568169644816863666886675"", ...",polymarket


In [47]:
# Cell 5 — Fetch Kalshi markets
def fetch_kalshi_markets(max_pages=50):  # increased from 10
    base_url = "https://external-api.kalshi.com/trade-api/v2/markets"
    all_markets = []
    cursor = None

    for _ in tqdm(range(max_pages), desc="Fetching Kalshi markets"):
        params = {"status": "settled", "limit": 1000}
        if cursor:
            params["cursor"] = cursor

        resp = requests.get(base_url, params=params)
        resp.raise_for_status()
        data = resp.json()

        markets = data.get("markets", [])
        all_markets.extend(markets)

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(0.5)

    print(f"Fetched {len(all_markets)} Kalshi markets")
    return all_markets

kalshi_markets_raw = fetch_kalshi_markets()

Fetching Kalshi markets: 100%|██████████| 50/50 [02:26<00:00,  2.93s/it]


Fetched 50000 Kalshi markets


In [48]:
# Cell 6 — Parse, filter, and save Kalshi markets
def parse_kalshi_market(m):
    return {
        "market_id": m.get("ticker"),
        "question": m.get("title"),
        "category": m.get("event_ticker"),
        "start_date": m.get("open_time"),
        "end_date": m.get("close_time"),
        "resolution_source": m.get("rules_primary"),
        "volume": m.get("volume_fp"),
        "liquidity": m.get("liquidity_dollars"),
        "outcome": m.get("result"),
        "platform": "kalshi"
    }

kalshi_markets = pd.DataFrame([parse_kalshi_market(m) for m in kalshi_markets_raw])
kalshi_markets["volume"] = pd.to_numeric(kalshi_markets["volume"], errors="coerce")

# Filter out KXMV parlay markets — they don't have standard trade records
kalshi_markets = kalshi_markets[~kalshi_markets["market_id"].str.startswith("KXMV")].copy()

kalshi_markets_filtered = kalshi_markets[kalshi_markets["volume"] > 100].copy()

kalshi_markets.to_parquet("../data/raw/markets_kalshi.parquet", index=False)
print(f"Total Kalshi markets (ex-parlays): {len(kalshi_markets)}")
print(f"Markets with volume > 100: {len(kalshi_markets_filtered)}")
print("\nSample tickers:")
print(kalshi_markets_filtered["market_id"].head(10).tolist())
kalshi_markets_filtered.head()

Total Kalshi markets (ex-parlays): 13
Markets with volume > 100: 12

Sample tickers:
['KXMLBTOTAL-26MAY301605SDWSH-14', 'KXMLBSPREAD-26MAY301610MIANYM-NYM7', 'KXMLBTOTAL-26MAY301605SDWSH-13', 'KXMLBSPREAD-26MAY301610MILHOU-HOU7', 'KXMLBSPREAD-26MAY301610MILHOU-HOU6', 'KXMLBSPREAD-26MAY301610MIANYM-NYM6', 'KXMLBSPREAD-26MAY301610MIANYM-NYM5', 'KXMLBSPREAD-26MAY301610LAATB-LAA8', 'KXMLBTOTAL-26MAY301605MINPIT-22', 'KXMLBTOTAL-26MAY301605MINPIT-21']


,market_id,question,category,start_date,end_date,resolution_source,volume,liquidity,outcome,platform
8077,KXMLBTOTAL-26MAY301605SDWSH-14,San Diego vs Washington Total Runs?,KXMLBTOTAL-26MAY301605SDWSH,2026-05-30T22:42:12Z,2026-05-30T22:54:49Z,If San Diego and Washington collectively score...,866.22,0.0000,no,kalshi
13541,KXMLBSPREAD-26MAY301610MIANYM-NYM7,Mets wins by over 6.5 runs?,KXMLBSPREAD-26MAY301610MIANYM,2026-05-30T22:32:02Z,2026-05-30T23:00:18Z,If Mets wins by more than 6.5 runs in the Miam...,571.00,0.0000,no,kalshi
17825,KXMLBTOTAL-26MAY301605SDWSH-13,San Diego vs Washington Total Runs?,KXMLBTOTAL-26MAY301605SDWSH,2026-05-30T22:23:31Z,2026-05-30T22:54:48Z,If San Diego and Washington collectively score...,10746.84,0.0000,yes,kalshi
28753,KXMLBSPREAD-26MAY301610MILHOU-HOU7,Astros wins by over 6.5 runs?,KXMLBSPREAD-26MAY301610MILHOU,2026-05-30T22:02:59Z,2026-05-30T23:10:29Z,If Astros wins by more than 6.5 runs in the Mi...,12781.18,0.0000,yes,kalshi
28867,KXMLBSPREAD-26MAY301610MILHOU-HOU6,Astros wins by over 5.5 runs?,KXMLBSPREAD-26MAY301610MILHOU,2026-05-30T22:02:59Z,2026-05-30T23:10:29Z,If Astros wins by more than 5.5 runs in the Mi...,2169.82,0.0000,yes,kalshi


In [53]:
# Cell 7 — Fetch Polymarket trades (high volume markets only)
def fetch_polymarket_trades(market_id, limit=500):
    base_url = "https://data-api.polymarket.com/trades"
    all_trades = []
    offset = 0

    while True:
        params = {"conditionId": market_id, "limit": limit, "offset": offset}
        resp = requests.get(base_url, params=params)
        if resp.status_code != 200:
            break
        data = resp.json()
        if not data:
            break
        all_trades.extend(data)
        if len(data) < limit:
            break
        offset += limit

    return all_trades

pm_markets_filtered = pm_markets[pm_markets["volume"] > 5000].copy()
print(f"Polymarket markets with volume > $5000: {len(pm_markets_filtered)}")

all_pm_trades = []
for market_id in tqdm(pm_markets_filtered["market_id"].dropna(), desc="Fetching Polymarket trades"):
    trades = fetch_polymarket_trades(market_id)
    for t in trades:
        t["market_id"] = market_id
    all_pm_trades.extend(trades)
    time.sleep(0.2)

pm_trades = pd.DataFrame(all_pm_trades) if all_pm_trades else pd.DataFrame()
pm_trades.to_parquet("../data/raw/trades_polymarket.parquet", index=False)
print(f"Saved {len(pm_trades)} Polymarket trades")

Polymarket markets with volume > $5000: 99


Fetching Polymarket trades: 100%|██████████| 99/99 [04:24<00:00,  2.67s/it]


Saved 346500 Polymarket trades


In [49]:
# Cell 8 — Fetch Kalshi trades
def fetch_kalshi_trades(ticker, limit=1000):
    base_url = "https://external-api.kalshi.com/trade-api/v2/markets/trades"
    all_trades = []
    cursor = None

    while True:
        params = {"ticker": ticker, "limit": limit}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(base_url, params=params)
        if resp.status_code != 200:
            break
        data = resp.json()
        trades = data.get("trades", [])
        all_trades.extend(trades)
        cursor = data.get("cursor")
        if not cursor:
            break

    return all_trades

sample_size = min(200, len(kalshi_markets_filtered))
kalshi_sample = kalshi_markets_filtered.sample(n=sample_size, random_state=42).copy()
print(f"Sampling {len(kalshi_sample)} markets")

all_kalshi_trades = []
for ticker in tqdm(kalshi_sample["market_id"].dropna(), desc="Fetching Kalshi trades"):
    trades = fetch_kalshi_trades(ticker)
    for t in trades:
        t["market_id"] = ticker
    all_kalshi_trades.extend(trades)
    time.sleep(0.1)

kalshi_trades = pd.DataFrame(all_kalshi_trades) if all_kalshi_trades else pd.DataFrame()
kalshi_trades.to_parquet("../data/raw/trades_kalshi.parquet", index=False)
print(f"Saved {len(kalshi_trades)} Kalshi trades")

Sampling 12 markets


Fetching Kalshi trades: 100%|██████████| 12/12 [00:06<00:00,  1.81it/s]

Saved 3640 Kalshi trades


In [54]:
# Cell 9 — Sanity check
print("=== Collection Summary ===")
print(f"Polymarket markets (total):    {len(pm_markets):,}")
print(f"Polymarket markets (filtered): {len(pm_markets_filtered):,}")
print(f"Kalshi markets (total):        {len(kalshi_markets):,}")
print(f"Kalshi markets (filtered):     {len(kalshi_markets_filtered):,}")
print(f"Polymarket trades:             {len(pm_trades):,}")
print(f"Kalshi trades:                 {len(kalshi_trades):,}")
print()
print("Polymarket trade columns:", list(pm_trades.columns))
print("Kalshi trade columns:    ", list(kalshi_trades.columns))

=== Collection Summary ===
Polymarket markets (total):    100
Polymarket markets (filtered): 99
Kalshi markets (total):        13
Kalshi markets (filtered):     12
Polymarket trades:             346,500
Kalshi trades:                 3,640

Polymarket trade columns: ['proxyWallet', 'side', 'asset', 'conditionId', 'size', 'price', 'timestamp', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'outcomeIndex', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'transactionHash', 'market_id']
Kalshi trade columns:     ['count_fp', 'created_time', 'no_price_dollars', 'taker_book_side', 'taker_outcome_side', 'taker_side', 'ticker', 'trade_id', 'yes_price_dollars', 'market_id']
